# Notebook 01 — Exploratory Data Analysis
**Project:** Credit Card Customer Segmentation  
**Dataset:** [Kaggle — Credit Card Dataset for Clustering](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata)  
**Goal:** Understand the data structure, distributions, missing values, and behavioural patterns before segmentation.

---
### Flow
1. Load & inspect data
2. Missing value analysis
3. Univariate distributions
4. Bivariate relationships
5. Correlation analysis
6. Key findings summary

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
df = load_raw_data()
df.head()

## 1. Data Overview

In [ ]:
print(f'Shape      : {df.shape}')
print(f'Duplicates : {df.duplicated().sum()}')
print(f'\nData Types:\n{df.dtypes}')

In [ ]:
df.describe().round(2)

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
print(missing_df)

# Visualize
plt.figure(figsize=(7, 3))
plt.barh(missing_df.index, missing_df['Missing %'], color='tomato', edgecolor='white')
for i, v in enumerate(missing_df['Missing %']):
    plt.text(v + 0.05, i, f'{v}%', va='center', fontsize=10)
plt.title('Missing Values (%)', fontweight='bold')
plt.xlabel('Percentage Missing')
plt.tight_layout()
plt.savefig('../reports/figures/00_missing_values.png', bbox_inches='tight')
plt.show()
print('\nStrategy: Impute with column median (robust to outliers)')

## 3. Univariate Distributions

In [ ]:
# Monetary features
monetary_cols = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
colors = ['steelblue', 'seagreen', 'tomato', 'orange', 'purple', 'brown']

for ax, col, color in zip(axes, monetary_cols, colors):
    ax.hist(df[col].dropna(), bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(df[col].median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Median: {df[col].median():,.0f}')
    ax.set_title(col)
    ax.set_xlabel('Value ($)')
    ax.legend(fontsize=8)

plt.suptitle('Monetary Feature Distributions (Right-Skewed)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_monetary_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Frequency features (0-1 scale)
freq_cols = ['BALANCE_FREQUENCY', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
             'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, freq_cols):
    ax.hist(df[col].dropna(), bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Frequency (0-1)')
    ax.set_xlim(0, 1)

plt.suptitle('Frequency Feature Distributions (0-1 Scale)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_frequency_distributions.png', bbox_inches='tight')
plt.show()

## 4. Bivariate Relationships

In [ ]:
# Key relationships: Balance vs Purchases, Credit Limit vs Payments, Cash Advance vs Balance
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(df['BALANCE'], df['PURCHASES'], alpha=0.2, s=8, color='steelblue')
axes[0].set_title('Balance vs Purchases')
axes[0].set_xlabel('Balance ($)')
axes[0].set_ylabel('Purchases ($)')

axes[1].scatter(df['CREDIT_LIMIT'], df['PAYMENTS'], alpha=0.2, s=8, color='seagreen')
axes[1].set_title('Credit Limit vs Payments')
axes[1].set_xlabel('Credit Limit ($)')
axes[1].set_ylabel('Payments ($)')

axes[2].scatter(df['CASH_ADVANCE'], df['BALANCE'], alpha=0.2, s=8, color='tomato')
axes[2].set_title('Cash Advance vs Balance')
axes[2].set_xlabel('Cash Advance ($)')
axes[2].set_ylabel('Balance ($)')

plt.suptitle('Key Feature Relationships', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_bivariate_relationships.png', bbox_inches='tight')
plt.show()

In [ ]:
# Full payment behaviour
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PRC_FULL_PAYMENT distribution
axes[0].hist(df['PRC_FULL_PAYMENT'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Full Payment Rate Distribution')
axes[0].set_xlabel('% of Full Payments')
axes[0].set_ylabel('Count')

# Tenure distribution
tenure_counts = df['TENURE'].value_counts().sort_index()
axes[1].bar(tenure_counts.index, tenure_counts.values, color='seagreen', edgecolor='white')
axes[1].set_title('Customer Tenure Distribution')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Count')

plt.suptitle('Payment Behaviour & Tenure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_payment_tenure.png', bbox_inches='tight')
plt.show()

full_payers = (df['PRC_FULL_PAYMENT'] == 1).sum()
non_payers  = (df['PRC_FULL_PAYMENT'] == 0).sum()
print(f'Always pay in full : {full_payers:,} ({full_payers/len(df)*100:.1f}%)')
print(f'Never pay in full  : {non_payers:,} ({non_payers/len(df)*100:.1f}%)')

## 5. Correlation Analysis

In [ ]:
df_num = df.drop(columns=['CUST_ID'])
corr = df_num.corr()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, square=True, cbar_kws={'shrink': 0.75}, annot_kws={'size': 7})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/05_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Top correlations
corr_pairs = corr.unstack().drop_duplicates()
corr_pairs = corr_pairs[corr_pairs != 1.0].sort_values(ascending=False)
print('Top 10 feature correlations:')
print(corr_pairs.head(10).round(3))

## 6. Key Findings

| Finding | Detail |
|---|---|
| **Missing values** | `MINIMUM_PAYMENTS` (3.5%), `CREDIT_LIMIT` (0.01%) — impute with median |
| **Right-skewed monetary features** | Most customers have low balances/purchases; a few have very high values |
| **Two purchase types** | One-off vs installment purchases are distinct behaviours |
| **Cash advance users** | ~50% of customers never use cash advances (strong segmentation signal) |
| **Full payment behaviour** | Most customers never pay in full (revolving credit users) |
| **Tenure mostly 12 months** | Dataset is dominated by 12-month customers |
| **High correlations** | PURCHASES & PURCHASES_TRX, CASH_ADVANCE & CASH_ADVANCE_TRX (expected) |
| **No categorical features** | All features are numerical — no encoding needed, focus on scaling |

> **Next step:** `02_Feature_Engineering.ipynb` — impute missing values, engineer interaction features, and scale for clustering.